# 🍷 CÔNG CỤ MÔ PHỎNG ĐỘ LỆCH DỮ LIỆU RƯỢU VANG TRONG CÁC ĐIỀU KIỆN MÔI TRƯỜNG
## VÀ TÁI XÁC LẬP ĐIỂM CHẤT LƯỢNG TIỀM NĂNG (POTENTIAL QUALITY POINTS)

---

### 🎯 Mục tiêu nghiên cứu & Ứng dụng
1. **Mô phỏng chân thực các biến động hóa lý:** Tác động của nhiệt độ cao, oxy hóa nút bần, lão hóa tự nhiên và sai số cảm biến đo lường.
2. **Tái thiết lập điểm chất lượng tiềm năng:** Ước lượng khoa học sự sụt giảm/thay đổi điểm số sau biến động môi trường.
3. **Kiểm định độ lệch phân phối (Data Drift):** Sử dụng kiểm định Kolmogorov-Smirnov và khoảng cách Wasserstein.
4. **An toàn dữ liệu tuyệt đối:** Hoàn toàn bảo toàn nguyên bản dữ liệu gốc `chatluong_ruougoc.csv`.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập đường dẫn thư mục gốc
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import doc_du_lieu_chuan
from src.cleaner import CAC_COT_HOA_LY
from src.environment_simulator import WineEnvironmentSimulator, KICH_BAN_MOI_TRUONG, tao_bo_du_lieu_da_moi_truong
from src.visualizer import thiet_lap_phong_cach_do_hoa

thiet_lap_phong_cach_do_hoa()
print("✅ Đã nạp thành công các module mô phỏng môi trường và dữ liệu!")


## 1. Nạp Dữ Liệu Sạch Nguyên Bản (Bảo Tồn File Gốc)
Chúng ta đọc bản sao dữ liệu sạch từ `data/duleu_chuan.feather`. File gốc thô `chatluong_ruougoc.csv` được giữ nguyên vẹn 100%.

In [ ]:
# Nạp dữ liệu chuẩn hóa
df_chuan = doc_du_lieu_chuan()
print(f"Tổng số mẫu dữ liệu: {len(df_chuan):,} mẫu")
print(f"Phân bố bậc chất lượng gốc:\n{df_chuan['bac_chat_luong'].value_counts()}")
df_chuan[["diem_chat_luong", "bac_chat_luong"] + CAC_COT_HOA_LY[:4]].head()


## 2. Khởi Tạo Công Cụ Mô Phỏng Môi Trường (WineEnvironmentSimulator)
Bộ công cụ sẽ tự động học quan hệ phi tuyến giữa 11 chỉ số hóa lý và điểm chất lượng gốc trên tập chuẩn.

In [ ]:
sim = WineEnvironmentSimulator(df_chuan, random_state=42)
print("Các kịch bản môi trường được hỗ trợ:")
for ma_kb, ten_kb in KICH_BAN_MOI_TRUONG.items():
    print(f" - [{ma_kb}]: {ten_kb}")


## 3. Khảo Sát Kịch Bản 1: Sốc Nhiệt & Bảo Quản Nhiệt Độ Cao (Heat Shock)
Nhiệt độ cao (>28°C - 35°C) làm tăng tốc độ oxy hóa, phân hủy SO₂ tự do và thúc đẩy hình thành axit bay hơi (mùi giấm).

In [ ]:
# Mô phỏng với cường độ biến động 50% (intensity = 0.5)
df_heat = sim.tao_du_lieu_lech_moi_truong(kich_ban="heat_shock", intensity=0.5)

# Xem 5 mẫu đối sánh điểm gốc và điểm tiềm năng tái xác lập
df_heat[[
    "diem_chat_luong_goc", "diem_chat_luong_tiem_nang", "delta_diem_chat_luong",
    "bac_chat_luong_goc", "bac_chat_luong_tiem_nang", "trang_thai_chuyen_doi"
]].head(8)


### Báo Cáo Kiểm Định Độ Lệch Phân Phối (Data Drift - Kolmogorov-Smirnov & Wasserstein)

In [ ]:
df_drift_heat = sim.tinh_toan_chi_so_drift(df_heat, df_chuan)
print("BẢNG KIỂM ĐỊNH DATA DRIFT (SẮP XẾP THEO MỨC BIẾN ĐỘNG):")
display(df_drift_heat)


### Ma Trận Chuyển Dịch Bậc Chất Lượng (Quality Transition Matrix)

In [ ]:
ma_tran, thong_ke = sim.tinh_ma_tran_chuyen_doi_chat_luong(df_heat)

print(f"Tỉ lệ mẫu bị tụt bậc: {thong_ke['ti_le_tut_hang_pct']}% ({thong_ke['so_mau_tut_hang']}/{thong_ke['tong_so_mau']} mẫu)")
print(f"Điểm trung bình gốc: {thong_ke['diem_goc_mean']} -> Điểm tiềm năng: {thong_ke['diem_tiem_nang_mean']} (Δ = {thong_ke['delta_diem_trung_binh']})")

plt.figure(figsize=(7, 5))
sns.heatmap(ma_tran, annot=True, fmt="d", cmap="YlOrRd", cbar=True, linewidths=1)
plt.title("Ma Trận Chuyển Dịch Chất Lượng Khi Sốc Nhiệt", fontweight="bold")
plt.xlabel("Bậc Tiềm Năng (Sau Biến Động)", fontweight="bold")
plt.ylabel("Bậc Nguyên Bản (Gốc)", fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Đối Sánh Tác Động Giữa Cả 4 Kịch Bản Môi Trường
Hãy cùng so sánh độ sụt giảm điểm số tiềm năng trung bình giữa: Sốc nhiệt, Hở nút bần oxy hóa, Lệch chuẩn cảm biến, và Lão hóa tự nhiên.

In [ ]:
danh_sach_so_sanh = []

for ma_kb, ten_kb in KICH_BAN_MOI_TRUONG.items():
    df_kb = sim.tao_du_lieu_lech_moi_truong(kich_ban=ma_kb, intensity=0.5)
    _, tk = sim.tinh_ma_tran_chuyen_doi_chat_luong(df_kb)
    danh_sach_so_sanh.append({
        "Kịch bản": ten_kb,
        "Điểm Gốc (Mean)": tk["diem_goc_mean"],
        "Điểm Tiềm Năng (Mean)": tk["diem_tiem_nang_mean"],
        "Δ Điểm Trung Bình": tk["delta_diem_trung_binh"],
        "Tỉ lệ tụt hạng (%)": tk["ti_le_tut_hang_pct"]
    })

df_tong_hop = pd.DataFrame(danh_sach_so_sanh).sort_values(by="Δ Điểm Trung Bình")
display(df_tong_hop)

# Trực quan hóa so sánh Δ Điểm
plt.figure(figsize=(10, 4.5))
sns.barplot(data=df_tong_hop, y="Kịch bản", x="Δ Điểm Trung Bình", palette="Reds_r")
plt.title("Mức Độ Suy Giảm Điểm Chất Lượng Tiềm Năng Theo Từng Môi Trường (Intensity = 0.5)", fontweight="bold")
plt.xlabel("Δ Điểm (Tiềm Năng - Gốc)")
plt.axvline(0, color="black", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


## 5. Thử Nghiệm Tinh Chỉnh Cường Độ Biến Động (Intensity Curve $\alpha \in [0.1, 1.0]$)
Khảo sát tốc độ suy giảm chất lượng khi điều kiện bảo quản ngày càng khắc nghiệt.

In [ ]:
cuong_do_range = np.linspace(0.1, 1.0, 10)
ket_qua_curve = []

for alpha in cuong_do_range:
    for ma_kb in ["heat_shock", "oxidation_cork_failure", "cellar_aging"]:
        df_test = sim.tao_du_lieu_lech_moi_truong(kich_ban=ma_kb, intensity=alpha)
        _, tk = sim.tinh_ma_tran_chuyen_doi_chat_luong(df_test)
        ket_qua_curve.append({
            "Intensity": alpha,
            "Kịch bản": KICH_BAN_MOI_TRUONG[ma_kb],
            "Tỉ lệ tụt hạng (%)": tk["ti_le_tut_hang_pct"],
            "Điểm tiềm năng (Mean)": tk["diem_tiem_nang_mean"]
        })

df_curve = pd.DataFrame(ket_qua_curve)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.lineplot(data=df_curve, x="Intensity", y="Điểm tiềm năng (Mean)", hue="Kịch bản", marker="o")
plt.title("Điểm Tiềm Năng Theo Cường Độ Môi Trường", fontweight="bold")
plt.xlabel("Cường độ biến động (Intensity)")
plt.ylabel("Điểm số trung bình")

plt.subplot(1, 2, 2)
sns.lineplot(data=df_curve, x="Intensity", y="Tỉ lệ tụt hạng (%)", hue="Kịch bản", marker="s")
plt.title("Tỉ Lệ Mẫu Tụt Hạng Theo Cường Độ Môi Trường", fontweight="bold")
plt.xlabel("Cường độ biến động (Intensity)")
plt.ylabel("Tỉ lệ tụt hạng (%)")

plt.tight_layout()
plt.show()


## 6. Xuất Dữ Liệu An Toàn & Độc Lập
Toàn bộ kết quả mô phỏng lệch môi trường được lưu vào các file riêng trong `data/`, tuyệt đối không ghi đè lên file thô ban đầu `data/chatluong_ruougoc.csv`.

In [ ]:
# Xuất file kịch bản Sốc nhiệt chuẩn để làm việc tiếp
file_xuat = sim.xuat_bao_cao_va_du_lieu_an_toan(df_heat, df_drift_heat, ma_tran)
print("✅ ĐÃ XUẤT CÁC FILE DỮ LIỆU ĐỘC LẬP THÀNH CÔNG:")
for loai, path in file_xuat.items():
    print(f" - {loai.upper()}: {path.resolve()}")
print("\n🛡️ FILE GỐC UCI: data/chatluong_ruougoc.csv ĐÃ ĐƯỢC BẢO VỆ NGUYÊN BẢN!")
